# 03 — Pré-processamento e Engenharia de Features

Este notebook realiza limpeza de dados, tratamento de valores ausentes,
codificação de variáveis categóricas e engenharia de features.

**Pré-requisito:** Execute o notebook `01_coleta_dados.ipynb` antes.

---

## 3.1 Importações

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
CAMINHO_DADOS = os.path.join("..", "dados", "bruto", "dataset_clientes.csv")
DIR_SAIDA = os.path.join("..", "dados", "processado")
os.makedirs(DIR_SAIDA, exist_ok=True)

## 3.2 Carregar Dados

In [ ]:
df = pd.read_csv(CAMINHO_DADOS)
print(f"Shape original: {df.shape}")
df.head()

## 3.3 Tratamento de Valores Ausentes

Estratégias:
- **`renda_mensal`**: Imputação pela **mediana** (distribuição assimétrica).
- **`satisfacao`**: Imputação pela **moda** (variável ordinal).

In [ ]:
print("Valores ausentes antes do tratamento:")
print(df.isnull().sum()[df.isnull().sum() > 0])

mediana_renda = df["renda_mensal"].median()
moda_satisfacao = df["satisfacao"].mode()[0]

df["renda_mensal"] = df["renda_mensal"].fillna(mediana_renda)
df["satisfacao"] = df["satisfacao"].fillna(moda_satisfacao)

print(f"\n→ renda_mensal imputada com mediana = {mediana_renda:.2f}")
print(f"→ satisfacao imputada com moda = {moda_satisfacao}")
print(f"\nValores ausentes restantes: {df.isnull().sum().sum()}")

## 3.4 Tratamento de Outliers (Winsorização P1–P99)

In [ ]:
for col in ["renda_mensal", "saldo"]:
    p1 = df[col].quantile(0.01)
    p99 = df[col].quantile(0.99)
    n_ajustados = ((df[col] < p1) | (df[col] > p99)).sum()
    df[col] = df[col].clip(lower=p1, upper=p99)
    print(f"{col}: {n_ajustados} valores ajustados para [{p1:.2f}, {p99:.2f}]")

## 3.5 Engenharia de Features

Features derivadas baseadas em hipóteses de negócio:

| Feature | Fórmula | Hipótese |
|---------|---------|----------|
| `renda_por_produto` | renda / num_produtos | Menor renda/produto → maior insatisfação |
| `reclamacao_por_tempo` | reclamações / (tempo + 1) | Alta taxa de reclamação recente → churn |
| `faixa_etaria` | bins de idade | Padrões geracionais diferentes |
| `cliente_premium` | saldo > med AND renda > med | Perfil de cliente de alto valor |

In [ ]:
# Renda por produto
df["renda_por_produto"] = df["renda_mensal"] / df["num_produtos"]

# Taxa de reclamação
df["reclamacao_por_tempo"] = df["num_reclamacoes"] / (df["tempo_conta_meses"] + 1)

# Faixa etária
df["faixa_etaria"] = pd.cut(
    df["idade"],
    bins=[0, 25, 35, 45, 55, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "56+"]
)

# Cliente premium
df["cliente_premium"] = (
    (df["saldo"] > df["saldo"].median()) &
    (df["renda_mensal"] > df["renda_mensal"].median())
).astype(int)

print(f"Shape após engenharia de features: {df.shape}")
df[["renda_por_produto", "reclamacao_por_tempo", "faixa_etaria", "cliente_premium"]].describe()

## 3.6 Codificação de Variáveis Categóricas (One-Hot Encoding)

In [ ]:
colunas_cat = ["sexo", "canal_aquisicao", "faixa_etaria"]
df = pd.get_dummies(df, columns=colunas_cat, drop_first=True, dtype=int)

print(f"Shape após encoding: {df.shape}")
print(f"\nColunas: {list(df.columns)}")

## 3.7 Divisão Treino/Teste e Normalização

In [ ]:
# Remover ID
df = df.drop(columns=["id_cliente"], errors="ignore")

X = df.drop(columns=["churn"])
y = df["churn"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Normalização
colunas_norm = ["idade", "renda_mensal", "tempo_conta_meses", "saldo",
                "num_reclamacoes", "satisfacao", "renda_por_produto", "reclamacao_por_tempo"]
colunas_norm = [c for c in colunas_norm if c in X_treino.columns]

scaler = StandardScaler()
X_treino[colunas_norm] = scaler.fit_transform(X_treino[colunas_norm])
X_teste[colunas_norm] = scaler.transform(X_teste[colunas_norm])

treino = pd.concat([X_treino, y_treino], axis=1)
teste = pd.concat([X_teste, y_teste], axis=1)

print(f"Treino: {treino.shape[0]} registros | Churn: {y_treino.mean():.2%}")
print(f"Teste:  {teste.shape[0]} registros | Churn: {y_teste.mean():.2%}")

## 3.8 Salvar Datasets Processados

In [ ]:
df.to_csv(os.path.join(DIR_SAIDA, "dataset_processado.csv"), index=False)
treino.to_csv(os.path.join(DIR_SAIDA, "dataset_treino.csv"), index=False)
teste.to_csv(os.path.join(DIR_SAIDA, "dataset_teste.csv"), index=False)
print("✓ Datasets processados salvos em dados/processados/")